# Statistical A/B test analysis

This notebook compares conversion rates for the advertisement and PSA groups using the validated SQLite experiment data.

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

database_path = Path.cwd() / "data" / "marketing_ab_test.db"
if not database_path.exists():
    database_path = Path.cwd().parent / "data" / "marketing_ab_test.db"

if not database_path.is_file():
    raise FileNotFoundError("Run sql/create_database.py before this notebook.")

with sqlite3.connect(database_path) as connection:
    experiment_df = pd.read_sql_query(
        "SELECT test_group, converted FROM experiment_data",
        connection,
    )

experiment_df.head()

In [ ]:
summary = (
    experiment_df.groupby("test_group")
    .agg(users=("converted", "size"), conversions=("converted", "sum"))
    .reindex(["ad", "psa"])
)
summary["conversion_rate"] = summary["conversions"] / summary["users"]
summary["conversion_rate_percent"] = summary["conversion_rate"] * 100
summary

## Hypotheses

**Null hypothesis:** The conversion rate for the advertisement group is equal to the conversion rate for the PSA group.

**Alternative hypothesis:** The conversion rate for the advertisement group is different from the conversion rate for the PSA group.

The significance level is **alpha = 0.05**.

In [ ]:
alpha = 0.05
conversions = summary["conversions"].to_numpy()
users = summary["users"].to_numpy()

z_statistic, p_value = proportions_ztest(count=conversions, nobs=users)

print(f"z-statistic: {z_statistic:.6f}")
print(f"p-value: {p_value:.6g}")

In [ ]:
ad_rate = summary.loc["ad", "conversion_rate"]
psa_rate = summary.loc["psa", "conversion_rate"]

absolute_lift = ad_rate - psa_rate
relative_lift = absolute_lift / psa_rate * 100

print(f"Absolute lift: {absolute_lift * 100:.6f} percentage points")
print(f"Relative lift: {relative_lift:.6f}%")

In [ ]:
standard_error = (
    ad_rate * (1 - ad_rate) / summary.loc["ad", "users"]
    + psa_rate * (1 - psa_rate) / summary.loc["psa", "users"]
) ** 0.5
margin_of_error = 1.96 * standard_error
confidence_interval = (
    (absolute_lift - margin_of_error) * 100,
    (absolute_lift + margin_of_error) * 100,
)

print(
    "95% confidence interval for ad rate - PSA rate: "
    f"({confidence_interval[0]:.6f}, {confidence_interval[1]:.6f}) percentage points"
)

In [ ]:
decision = "Reject the null hypothesis" if p_value < alpha else "Fail to reject the null hypothesis"
print(f"Decision at alpha = {alpha}: {decision}")

print(
    f"The ad group had a higher conversion rate ({ad_rate * 100:.4f}%) than the "
    f"PSA group ({psa_rate * 100:.4f}%). The absolute lift was "
    f"{absolute_lift * 100:.4f} percentage points, equivalent to a "
    f"{relative_lift:.2f}% relative lift. The difference was statistically "
    f"significant at the 0.05 level (p = {p_value:.6g})."
)
print(
    "This result suggests that the advertisement group performed better in this "
    "dataset, but it does not by itself prove that advertising caused the improvement. "
    "The groups are highly unequal in size, and the original randomization procedure "
    "was not independently verified in this project."
)